In [4]:
import pandas as pd
import numpy as np

In [5]:
ganjoo_bbb = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\bbb\t20_bbb.csv", low_memory=False)

In [6]:
ganjoo_bbb.columns

Index(['p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl',
       'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal',
       'p_out', 'over', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs',
       'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs',
       'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem',
       'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'date',
       'year', 'ground', 'country', 'winner', 'toss', 'competition',
       'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced',
       'bowlruns', 'bat_out', 'wagonX', 'wagonY', 'wagonZone', 'line',
       'length', 'shot', 'control', 'predscore', 'wprob'],
      dtype='object')

In [7]:
bathand_map = ganjoo_bbb[['p_bat', 'bat', 'bat_hand']].drop_duplicates()
bowlstyle_map = ganjoo_bbb[['p_bowl', 'bowl', 'bowl_style']].drop_duplicates()

In [8]:
player_detail = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\player_details.csv")
people = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\people3.csv")

In [9]:
bb_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\bbb\all_league_all_matches_bbb.csv")
xi_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\all_league_playing_xi.csv")

C:\Users\HP\AppData\Local\Temp\ipykernel_13792\3252331213.py:1: DtypeWarning: Columns (1,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  bb_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\bbb\all_league_all_matches_bbb.csv")


In [10]:
# Convert player IDs in playing XI from string representation to actual lists of integers.
for col in ['team1_playing_xi', 'team2_playing_xi']:
    xi_df[col] = xi_df[col].apply(
        lambda x:
            [int(p.strip('"'))
             for p in x.strip('[]').replace(' ', '').split(',')
             if p.strip('"') != '']
            if isinstance(x, str) else []
    )

Making the columns compatible to the code

In [11]:
bb_df.rename(columns={
    'start_date': 'date',
    'innings': 'inns',
    'batting_team': 'team_bat',
    'bowling_team': 'team_bowl',
    'striker': 'batter',
    'runs_off_bat': 'batruns'
}, inplace=True)

drop_cols = ['season', 'non_striker', 'extras', 'penalty',
             'other_wicket_type', 'other_player_dismissed', 'league']
bb_df.drop(columns=drop_cols, inplace=True, errors='ignore')

bb_df.wides = bb_df.wides.fillna(0)
bb_df.noballs = bb_df.noballs.fillna(0)
bb_df.byes = bb_df.byes.fillna(0)
bb_df.legbyes = bb_df.legbyes.fillna(0)

In [12]:
name_to_id = dict(zip(people['unique_name'], people['key_cricinfo']))
id_to_batrole = dict(zip(player_detail['key_cricinfo'], player_detail['bat_style']))
id_to_bowlrole = dict(zip(player_detail['key_cricinfo'], player_detail['bowl_style']))
bb_df['player_dismissed'] = bb_df.player_dismissed.map(name_to_id)
bb_df['batter_id'] = bb_df['batter'].map(name_to_id)
bb_df['bowler_id'] = bb_df['bowler'].map(name_to_id)
bb_df['bat_hand'] = bb_df['batter_id'].map(id_to_batrole)
bb_df['bowl_style'] = bb_df['bowler_id'].map(id_to_bowlrole)
bb_df['ballfaced'] = bb_df.apply(lambda row: 0 if (row['wides'] == 1 or row['noballs'] == 1) else 1, axis=1)
bb_df['bowlruns'] = bb_df['batruns'] + bb_df['wides'] + bb_df['noballs']
# Reorder columns to match IPL data ordering
ipl_columns = ['date', 'match_id', 'inns', 'ball', 'batter', 'batter_id',
               'team_bat', 'bowler', 'bowler_id', 'team_bowl', 'batruns',
               'bowlruns', 'ballfaced', 'wides', 'noballs', 'byes', 'legbyes',
               'wicket_type', 'player_dismissed', 'bat_hand', 'bowl_style', 'venue']

# Reorder the DataFrame columns
bb_df = bb_df[ipl_columns]

In [13]:
bb_df.bat_hand.isna().sum(), bb_df.bowl_style.isna().sum(), bb_df.batter_id.isna().sum(), bb_df.bowler_id.isna().sum()

(862673, 869851, 0, 0)

In [14]:
batmap = dict(zip(bathand_map.p_bat, bathand_map.bat_hand))
bowlmap = dict(zip(bowlstyle_map.p_bowl, bowlstyle_map.bowl_style))
# Update 'bat_hand' only when it is null
mask_bat = bb_df['bat_hand'].isnull()
bb_df.loc[mask_bat, 'bat_hand'] = bb_df.loc[mask_bat, 'batter_id'].map(batmap)
bb_df.loc[mask_bat, 'bat_hand'] = bb_df.loc[mask_bat, 'batter_id'].fillna('RHB')

# Update 'bowl_style' only when it is null
mask_bowl = bb_df['bowl_style'].isnull()
bb_df.loc[mask_bowl, 'bowl_style'] = bb_df.loc[mask_bowl, 'bowler_id'].map(bowlmap)
bb_df.loc[mask_bowl, 'bowl_style'] = bb_df.loc[mask_bowl, 'bowler_id'].fillna('RAF')

In [15]:
bb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1689221 entries, 0 to 1689220
Data columns (total 22 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   date              1689221 non-null  object 
 1   match_id          1689221 non-null  int64  
 2   inns              1689221 non-null  int64  
 3   ball              1689221 non-null  float64
 4   batter            1689221 non-null  object 
 5   batter_id         1689221 non-null  float64
 6   team_bat          1689221 non-null  object 
 7   bowler            1689221 non-null  object 
 8   bowler_id         1689221 non-null  float64
 9   team_bowl         1689221 non-null  object 
 10  batruns           1689221 non-null  int64  
 11  bowlruns          1689221 non-null  float64
 12  ballfaced         1689221 non-null  int64  
 13  wides             1689221 non-null  float64
 14  noballs           1689221 non-null  float64
 15  byes              1689221 non-null  float64
 16  

In [16]:
print("Identifying player name inconsistencies...")
# Define the corrections: {batter_id: 'Correct Name'}
name_corrections = {
    4864: 'Dan Christian',          # Choosing the more common name
    27280: 'Manvinder Bisla',        # Removing extra space
    302208: 'CM Gautam',             # Choosing the shorter/common name
    322535: 'Karanveer Singh',       # Choosing the likely correct spelling
    326016: 'Bhuvneshwar Kumar',     # Correct spelling
    784379: 'Wanindu Hasaranga'      # Shorter/common name
}
print(f"Applying corrections for {len(name_corrections)} player IDs...")
corrections_applied_count = 0
# Loop through the identified player IDs and apply the standard name
for player_id, correct_name in name_corrections.items():
    # Create a mask for rows matching the current player ID
    mask = bb_df['batter_id'] == player_id

    # Check if any rows match before attempting update
    if mask.any():
        # Update the 'batter' column for these rows to the correct name
        bb_df.loc[mask, 'batter'] = correct_name
        corrections_applied_count += 1
        # Optional: You could print which IDs were updated
        # print(f"  - Updated batter name for ID {player_id} to '{correct_name}'")
    else:
        print(f"  - Warning: No rows found in bb_df for batter_id {player_id}. No correction applied.")

print(f"Name corrections applied for {corrections_applied_count} IDs found in the DataFrame.")

Identifying player name inconsistencies...
Applying corrections for 6 player IDs...
Name corrections applied for 6 IDs found in the DataFrame.


In [17]:
bb_df['dismissal'] = bb_df.apply(lambda row: 1 if pd.notnull(row['player_dismissed']) and row['player_dismissed'] == row['batter_id'] else 0, axis=1)

### TASK 1 - Batter Performance Metrics

In [18]:
# --- Step 0: Create helper mappings using bb_df ---

# Mapping for batter names: use the first non-null occurrence per batter_id.
batter_name_map = bb_df.dropna(subset=['batter']).groupby('batter_id')['batter'].first().to_dict()

# Mapping for match date: assume each match_id in bb_df has a consistent date, so we take the first date seen.
match_date_map = bb_df.groupby('match_id')['date'].first().to_dict()

# --- Step 1: Prepare the ball-by-ball data ---
# Create a dismissal indicator column: if the 'player_dismissed' equals 'batter_id', mark as 1, else 0.
bb_df['dismissal'] = bb_df.apply(
    lambda row: 1 if pd.notnull(row['player_dismissed']) and (row['player_dismissed'] == row['batter_id']) else 0,
    axis=1
)

# Define helper functions to identify boundaries.
def is_four(run):
    return 1 if run == 4 else 0

def is_six(run):
    return 1 if run == 6 else 0

# Aggregate ball-by-ball data for each match, batter and bowler type.
agg_bb = bb_df.groupby(['match_id', 'batter_id', 'bowl_style']).agg(
    total_runs=('batruns', 'sum'),
    total_balls=('ballfaced', 'sum'),
    fours=('batruns', lambda x: x.apply(is_four).sum()),
    sixes=('batruns', lambda x: x.apply(is_six).sum()),
    dismissals=('dismissal', lambda x: 1 if x.sum() >= 1 else 0) # Count dismissal as 1 for the match if any record shows a dismissal for that batter.
).reset_index()

# --- Step 2: Create a DataFrame of all match/player-bowl_style combinations ---
# Here, we use xi_df which should contain a column 'match_id', and columns for both teams' playing XI:
# 'team1_playing_xi' and 'team2_playing_xi'.
rows = []
for i, row in xi_df.iterrows():
    match_id = row['match_id']
    # If you have a date in xi_df, you could use it to overwrite the bb_df date mapping.
    # Here we rely on match_date_map obtained from bb_df.
    players = row['team1_playing_xi'] + row['team2_playing_xi']
    for batter in players:
        batter_int = int(batter)  # convert to int to ensure matching with bb_df's batter_id
        for style in ['RAF', 'LAF', 'RLB', 'ROB', 'SLA', 'LWS']:
            rows.append({'match_id': match_id, 'batter_id': batter_int, 'bowl_style': style})

all_combinations = pd.DataFrame(rows)

# Merge the aggregated ball-by-ball stats with all combinations.
match_stats = pd.merge(all_combinations, agg_bb, on=['match_id', 'batter_id', 'bowl_style'], how='left')
# Fill missing stats (when a batter did not bat against a bowling style) with 0.
match_stats[['total_runs', 'total_balls', 'fours', 'sixes', 'dismissals']] = match_stats[['total_runs', 'total_balls', 'fours', 'sixes', 'dismissals']].fillna(0)

# Add the date for each match from the mapping.
match_stats['date'] = match_stats['match_id'].map(match_date_map)

# --- Step 3: Compute cumulative performance metrics ---
# Sort match_stats by batter, bowling style and match (here using match_id order).
match_stats = match_stats.sort_values(['batter_id', 'bowl_style', 'match_id'])

def compute_cumulative_metrics(grp):
    rpb_list = []
    bdr_pct_list = []
    rpd_list = []
    bpd_list = []

    cum_runs = 0
    cum_balls = 0
    cum_fours = 0
    cum_sixes = 0
    cum_dismissals = 0

    for idx, row in grp.iterrows():
        # Compute metrics from all previous matches (if any).
        if cum_balls > 0:
            rpb = cum_runs / cum_balls
            bdr_pct = ((cum_fours + cum_sixes) / cum_balls) * 100
        else:
            rpb = 0
            bdr_pct = 0

        if cum_dismissals > 0:
            rpd = cum_runs / cum_dismissals
            bpd = cum_balls / cum_dismissals
        else:
            rpd = cum_runs if cum_runs > 20 else cum_runs*1.5
            bpd = cum_balls if cum_balls > 10 else cum_balls*1.5

        rpb_list.append(rpb)
        bdr_pct_list.append(bdr_pct)
        rpd_list.append(rpd)
        bpd_list.append(bpd)

        # Update cumulative stats with the current match data.
        cum_runs += row['total_runs']
        cum_balls += row['total_balls']
        cum_fours += row['fours']
        cum_sixes += row['sixes']
        cum_dismissals += row['dismissals']

    grp = grp.copy()
    grp['rpb'] = rpb_list
    grp['bdr_pct'] = bdr_pct_list
    grp['rpd'] = rpd_list
    grp['bpd'] = bpd_list
    return grp

result_df = match_stats.groupby(['batter_id', 'bowl_style'], group_keys=False).apply(compute_cumulative_metrics)

# --- Step 4: Add Batter Name ---
# Map the batter_id to batter name using batter_name_map.
result_df['batter'] = result_df['batter_id'].map(batter_name_map)

# Reorder columns as desired:
result_df = result_df[['match_id', 'date', 'batter_id', 'batter', 'bowl_style', 'rpb', 'bdr_pct', 'rpd', 'bpd']]

# Reset index if needed and sort.
result_df.reset_index(drop=True, inplace=True)
result_df.sort_values(by=['date','batter_id', 'bowl_style'], inplace=True)

# The playing XI files has data from IPL 2025 as well, so to drop such rows remove rows where date is null
# This works because the match_stats DataFrame has a date column that is filled with dates from bb_df.
result_df.dropna(subset=['date'],inplace=True)

# --- Step 5: Z-score Standardization and Batter Performance Metrics Calculation---
# List of metrics and their corresponding weights
metrics = {
    'rpb': 0.35,
    'bdr_pct': 0.2,
    'rpd': 0.25,
    'bpd': 0.2
}
# Create a copy of the dataframe or work on it directly if you don't need the original metrics
df_bat = result_df.copy()

# Standardize each metric column using Z-score standardization:
for col in metrics.keys():
    mean_val = df_bat[col].mean()
    std_val = df_bat[col].std()
    # Create a new column for standardized value
    df_bat[col + '_z'] = (df_bat[col] - mean_val) / std_val

# Compute batter quality score using the weighted sum of the standardized metrics:
# batter_quality_score = 0.3 * rpb_z + 0.3 * bdr_pct_z + 0.2 * rpd_z + 0.2 * bpd_z
df_bat['batter_perf_metric'] = (metrics['rpb'] * df_bat['rpb_z'] +
                              metrics['bdr_pct'] * df_bat['bdr_pct_z'] +
                              metrics['rpd'] * df_bat['rpd_z'] +
                              metrics['bpd'] * df_bat['bpd_z'])

# Drop the intermediate standardized columns if you no longer need them:
df_bat.drop(columns=[col+'_z' for col in metrics.keys()], inplace=True)

In [19]:
df_bat[df_bat.batter_id == 5334].tail(7)

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric
9448,1386106,2023-12-23,5334,AJ Finch,SLA,1.230958,14.496314,22.772727,18.500000,0.597927
7639,1386129,2024-01-13,5334,AJ Finch,LAF,1.323424,19.262782,41.222222,31.148148,1.192734
8001,1386129,2024-01-13,5334,AJ Finch,LWS,1.012500,8.750000,10.125000,10.000000,0.063655
8363,1386129,2024-01-13,5334,AJ Finch,RAF,1.336982,18.334636,33.529412,25.078431,0.980445
8725,1386129,2024-01-13,5334,AJ Finch,RLB,1.378323,17.791411,21.741935,15.774194,0.682953
9087,1386129,2024-01-13,5334,AJ Finch,ROB,1.395556,18.888889,44.857143,32.142857,1.289699
9449,1386129,2024-01-13,5334,AJ Finch,SLA,1.230958,14.496314,22.772727,18.500000,0.597927


### TASK 2 - Computing Bowler Quality Scores

In [20]:
bowl_df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\all_league\all_league_bowl_final.csv")
# Convert the 'overs' column to numeric (float) in the DataFrame before grouping.
bowl_df['overs'] = pd.to_numeric(bowl_df['overs'], errors='coerce')
# --- Preparatory Steps ---

# Ensure date is in datetime format if not already.
bowl_df['date'] = pd.to_datetime(bowl_df['date'])

# Sort bowl_df by bowler and date so that historical metrics are computed in order.
bowl_df = bowl_df.sort_values(['bowler_id', 'date'])

# --- Step 1: Aggregate & compute cumulative historical stats for each bowler ---
# First, group the scorecard by match and bowler_id so that we get one row per bowler per match.
match_bowler = bowl_df.groupby(['match_id', 'bowler_id', 'bowler', 'date']).agg({
    'runs': 'sum',
    'wickets': 'sum',
    'overs': 'sum',
    'balls': 'sum'
}).reset_index()

# For cumulative stats we need a match counter per bowler.
match_bowler['match_count'] = 1

# Compute cumulative statistics for each bowler (each row's metrics are computed only using matches BEFORE the current match)
def compute_cumulative_bowling(grp):
    cum_runs = 0
    cum_wickets = 0
    cum_overs = 0
    cum_balls = 0
    cum_matches = 0

    avg_list = []
    sr_list = []
    wp_match_list = []
    op_match_list = []
    cum_match_list = []

    # Iterate row by row for this bowler (already sorted by date)
    for idx, row in grp.iterrows():
        # Use historical cumulative stats from previous matches to compute metrics.
        if cum_wickets > 0:
            # Bowling average: runs conceded per wicket (lower is better)
            avg = cum_runs / cum_wickets
            # Strike rate: balls per wicket (lower is better)
            sr = cum_balls / cum_wickets
        else:
            avg = cum_runs if cum_runs > 30 else cum_runs * 1.4
            sr = cum_balls if cum_balls > 25 else cum_balls * 1.4

        # Wickets per match (higher is better)
        if cum_matches > 0:
            wp_match = cum_wickets / cum_matches
            op_match = cum_overs / cum_matches
        else:
            wp_match = np.nan
            op_match = np.nan

        avg_list.append(avg)
        sr_list.append(sr)
        wp_match_list.append(wp_match)
        op_match_list.append(op_match)
        cum_match_list.append(cum_matches)

        # Update cumulative stats with current match data (so that next row uses these)
        cum_runs += row['runs']
        cum_wickets += row['wickets']
        cum_overs += row['overs']
        cum_balls += row['balls']
        cum_matches += 1

    grp = grp.copy()
    grp['bowling_avg_hist'] = avg_list
    grp['strike_rate_hist'] = sr_list
    grp['wickets_per_match'] = wp_match_list
    grp['overs_per_match'] = op_match_list
    grp['matches_played'] = cum_match_list
    return grp

cum_bowl = match_bowler.groupby('bowler_id', group_keys=False).apply(compute_cumulative_bowling)

# Keep only rows with historical stats already computed.
cum_bowl[['bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match']] = \
    cum_bowl[['bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match']].fillna(0)

# --- Step 2: Normalization of Metrics with Inverse/Direct Transformations ---

# Define a helper function for min–max normalization.
def min_max_norm(series, inverse=False):
    min_val = series.min()
    max_val = series.max()
    # If all values are the same:
    if max_val == min_val:
        return series.apply(lambda x: 0.0 if x == 0 else 1.0)
    if inverse:
        return (max_val - series) / (max_val - min_val)
    else:
        return (series - min_val) / (max_val - min_val)

# Normalized metrics:
cum_bowl['norm_avg'] = min_max_norm(cum_bowl['bowling_avg_hist'], inverse=True)   # lower is better
cum_bowl['norm_sr']  = min_max_norm(cum_bowl['strike_rate_hist'], inverse=True)    # lower is better
cum_bowl['norm_wp']  = min_max_norm(cum_bowl['wickets_per_match'], inverse=False)  # higher is better
cum_bowl['norm_op']  = min_max_norm(cum_bowl['overs_per_match'], inverse=False)    # higher is better

# --- Step 3: Compute the Bowling Quality Score Using Weights ---
# Weights: Bowling Average: 0.2, Strike Rate: 0.2, Wickets per Match: 0.3, Overs per Match: 0.3
cum_bowl['bowler_quality_score'] = (0.2 * cum_bowl['norm_avg'] +
                                    0.2 * cum_bowl['norm_sr'] +
                                    0.3 * cum_bowl['norm_wp'] +
                                    0.3 * cum_bowl['norm_op'])

# Apply credibility adjustment:
# Suppose full credibility is given after 45 matches.
credibility_weight = np.minimum(1, cum_bowl['matches_played'] / 45)
cum_bowl['bowler_quality_score'] = cum_bowl['bowler_quality_score'] * credibility_weight

# --- Finalize the DataFrame ---
final_bowler_df = cum_bowl[['match_id', 'date', 'bowler_id', 'bowler',
                             'bowling_avg_hist', 'strike_rate_hist', 'wickets_per_match', 'overs_per_match',
                             'bowler_quality_score']].copy()

# Penalize part-time or non-bowlers.
def adjust_for_part_time(row):
    if row['overs_per_match'] < 0.4:
        return row['bowler_quality_score'] * (row['overs_per_match'] / 0.4)
    else:
        return row['bowler_quality_score']

final_bowler_df.loc[:, 'bowler_quality_score'] = final_bowler_df.apply(adjust_for_part_time, axis=1)

# --- Final Normalization Step ---
# Re-normalize bowler_quality_score so that its min is 0 and max is 1.
min_score = final_bowler_df['bowler_quality_score'].min()
max_score = final_bowler_df['bowler_quality_score'].max()
final_bowler_df.loc[:, 'bowler_quality_score'] = (final_bowler_df['bowler_quality_score'] - min_score) / (max_score - min_score)

In [21]:
final_bowler_df

,match_id,date,bowler_id,bowler,bowling_avg_hist,strike_rate_hist,wickets_per_match,overs_per_match,bowler_quality_score
0,211028,2005-06-13,4578,MJ Clarke,0.000000,0.000000,0.000000,0.000000,0.000000
1,211028,2005-06-13,5390,AC Gilchrist,0.000000,0.000000,0.000000,0.000000,0.000000
2,211028,2005-06-13,5392,JN Gillespie,0.000000,0.000000,0.000000,0.000000,0.000000
3,211028,2005-06-13,5616,ML Hayden,0.000000,0.000000,0.000000,0.000000,0.000000
4,211028,2005-06-13,5939,MEK Hussey,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
161036,1476798,2025-03-23,1193293,Akhil Kumar,12.875000,11.875000,1.000000,1.925000,0.136855
161037,1476798,2025-03-23,1206188,K Tathgur,0.000000,0.000000,0.000000,0.000000,0.000000
161038,1476798,2025-03-23,1206190,S Movva,0.000000,0.000000,0.000000,0.000000,0.000000
161039,1476798,2025-03-23,1229251,DS Bajwa,28.000000,14.000000,0.083333,0.166667,0.056026


### TASK 3 - Compute Bowler Type Proportion in Opposition

In [22]:
# Step 1: Setup mapping from bowler_id to bowl_style
bowler_style_map = bb_df.dropna(subset=['bowl_style']) \
                         .groupby('bowler_id')['bowl_style'] \
                         .first().to_dict()

final_bowler_df['bowl_style'] = final_bowler_df['bowler_id'].map(bowler_style_map)
final_bowler_df['bowl_style'] = final_bowler_df['bowl_style'].fillna('Unknown')

# Step 2: Get opposition team
def get_opposition_team(match_id, batter_id):
    row = xi_df[xi_df['match_id'] == match_id].iloc[0]
    team1 = [int(x) for x in row['team1_playing_xi']]
    team2 = [int(x) for x in row['team2_playing_xi']]
    if batter_id in team1:
        return team2
    elif batter_id in team2:
        return team1
    else:
        return []

# Step 3: Compute normalized bowler type proportions per batter per match
def compute_bowler_proportions_for_batter(match_id, batter_id):
    opposition_ids = get_opposition_team(match_id, batter_id)

    if not opposition_ids:
        return {}

    opp_bowlers = final_bowler_df[(final_bowler_df['match_id'] == match_id) &
                                  (final_bowler_df['bowler_id'].isin(opposition_ids))]

    if opp_bowlers.empty:
        return {}

    # Group by bowl_style and sum quality scores
    style_quality = opp_bowlers.groupby('bowl_style')['bowler_quality_score'].sum()

    total_quality = style_quality.sum()
    if total_quality == 0:
        return {}

    # Normalize so proportions sum to 1
    normalized_proportions = (style_quality / total_quality).to_dict()
    return normalized_proportions

# Step 4: Apply to df_bat
# For performance, we'll first compute a lookup dictionary
from collections import defaultdict

# (match_id, batter_id) → dict of bowl_style proportions
batter_match_to_proportion = {}

for (match_id, batter_id) in df_bat[['match_id', 'batter_id']].drop_duplicates().itertuples(index=False):
    batter_match_to_proportion[(match_id, batter_id)] = compute_bowler_proportions_for_batter(match_id, batter_id)

# Step 5: Use this to fill in the proportion column for the row's target bowl_style
def get_proportion(row):
    props = batter_match_to_proportion.get((row['match_id'], row['batter_id']), {})
    return props.get(row['bowl_style'], 0.0)

df_bat['bowler_proportion'] = df_bat.apply(get_proportion, axis=1)

In [23]:
df_bat.sample()

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric,bowler_proportion
204083,804709,2015-08-29,51782,FH Edwards,LWS,0.0,0.0,0.0,0.0,-0.929758,0.0


In [24]:
df_bat[df_bat.match_id == 1216510]

,match_id,date,batter_id,batter,bowl_style,rpb,bdr_pct,rpd,bpd,batter_perf_metric,bowler_proportion
7537,1216510,2020-09-24,5334,AJ Finch,LAF,1.340187,20.373832,44.812500,33.437500,1.308810,0.216833
7899,1216510,2020-09-24,5334,AJ Finch,LWS,0.968750,6.250000,7.750000,8.000000,-0.071919,0.000000
8261,1216510,2020-09-24,5334,AJ Finch,RAF,1.346803,18.567775,32.912500,24.437500,0.971628,0.391945
8623,1216510,2020-09-24,5334,AJ Finch,RLB,1.441077,18.518519,26.750000,18.562500,0.843194,0.236480
8985,1216510,2020-09-24,5334,AJ Finch,ROB,1.593548,22.903226,44.909091,28.181818,1.411749,0.154741
...,...,...,...,...,...,...,...,...,...,...,...
1026033,1216510,2020-09-24,1175441,Ravi Bishnoi,LWS,0.000000,0.000000,0.000000,0.000000,-0.929758,0.000000
1026160,1216510,2020-09-24,1175441,Ravi Bishnoi,RAF,0.250000,0.000000,1.500000,6.000000,-0.687222,0.639802
1026287,1216510,2020-09-24,1175441,Ravi Bishnoi,RLB,0.000000,0.000000,0.000000,0.000000,-0.929758,0.171852
1026414,1216510,2020-09-24,1175441,Ravi Bishnoi,ROB,0.000000,0.000000,0.000000,0.000000,-0.929758,0.158782


In [25]:
df_bat.batter_perf_metric.max(), df_bat.batter_perf_metric.min(), df_bat.batter_perf_metric.mean(), df_bat.batter_perf_metric.std()

(8.61673647726336,
 -0.9297582726011505,
 -1.7481233871583396e-16,
 0.8807054667479538)

### TASK 4 - Compute Batter Matchup Score

In [26]:
# --- Task 4: Compute Batter Matchup Score ---

# Step 1: For each batter (per match) compute the weighted performance metric:
#         weighted_perf = sum over bowler types (bowler_proportion * batter_perf_metric)
weighted_perf_df = (
    df_bat.groupby(['match_id', 'batter_id', 'batter'], as_index=False)
          .apply(lambda g: pd.Series({'weighted_perf': (g['bowler_proportion'] * g['batter_perf_metric']).sum()}))
)
weighted_perf_df.reset_index(drop=True, inplace=True)

# Step 2: Linearly map the weighted_perf values to a 1–10 scale.
#         That is, apply a min–max normalization where min value maps to 1 and max to 10.
min_val = weighted_perf_df['weighted_perf'].min()
max_val = weighted_perf_df['weighted_perf'].max()

# In case max_val equals min_val (degenerate case), we can simply assign 1 (or any constant) to all.
if max_val == min_val:
    weighted_perf_df['batter_matchup_score'] = 1.0
else:
    weighted_perf_df['batter_matchup_score'] = 1 + 9 * (weighted_perf_df['weighted_perf'] - min_val) / (max_val - min_val)

weighted_perf_df.drop(columns=['weighted_perf'], inplace=True)

In [27]:
weighted_perf_df

,match_id,batter_id,batter,batter_matchup_score
0,211028,4578,MJ Clarke,2.383368
1,211028,5390,AC Gilchrist,2.383368
2,211028,5392,JN Gillespie,2.383368
3,211028,5616,ML Hayden,2.383368
4,211028,5939,MEK Hussey,2.383368
...,...,...,...,...
160305,1476798,1193293,Akhil Kumar,2.383368
160306,1476798,1206188,K Tathgur,2.383368
160307,1476798,1206190,S Movva,2.383368
160308,1476798,1229251,DS Bajwa,2.383368


In [28]:
import pandas as pd

# Read the reference DataFrame and get unique keys.
df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\all_league_bat_features.csv')
z = df[['match_id', 'batter_id']].drop_duplicates()

# Get the weighted performance scores with unique key pairs.
x = weighted_perf_df[['match_id', 'batter_id', 'batter_matchup_score']].drop_duplicates()

# Left merge using keys from z to ensure the same number of rows.
merged_df = z.merge(x, on=['match_id', 'batter_id'], how='left')

# Fill missing matchup scores with 0.
merged_df['batter_matchup_score'] = merged_df['batter_matchup_score'].fillna(0)

# Now merged_df has the same unique key pairs as z.
print(df.shape, merged_df.shape, z.shape)

(161026, 35) (161025, 3) (161025, 2)


In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\all_league_bat_features.csv')

# Identify duplicate rows based on match_id and batter_id (keeping first occurrence)
duplicates = df[df[['match_id', 'batter_id']].duplicated(keep='first')]

print("The duplicate row(s) that are dropped:")
duplicates

The duplicate row(s) that are dropped:


,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
76840,1130677,2018-01-09,2,KV Sharma,30288,Railways,Vidarbha,Shaheed Veer Narayan Singh International Stadium,SMA,15.166667,...,30.0,15.0,187.5,2.0,8.0,11.666667,125.0,7.0,5.6,0.8


In [32]:
merged_df

,match_id,batter_id,batter_matchup_score
0,1157105,1175501,1.436181
1,1157157,1175501,1.728216
2,1205739,1175501,2.088885
3,1205757,1175501,1.958864
4,1205783,1175501,1.196831
...,...,...,...
161020,1076004,43871,2.287708
161021,1076006,43871,2.568854
161022,1076009,43871,2.855787
161023,478279,43860,2.041244


In [31]:
df.shape, weighted_perf_df.shape,merged_df.shape

((161026, 35), (160310, 4), (161025, 3))

### Exporting it to a CSV File

In [33]:
merged_df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\all_league_bat_matchup.csv", index=False)